# 05 - Product-Level and Customer-Activity Analytics
## Objectives 5 and 6

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

**Objective 5** ranks products by rating and by review count. A minimum-review threshold is
applied first, otherwise a product with three five-star reviews tops the table, and the review
count is printed beside every rank so the reader can weigh it.

**Objective 6** measures how review activity is spread across reviewers.

In [ ]:
import os, sys, glob

try:
    from da_common import *
except ModuleNotFoundError:
    _paths = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
        os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
    for _p in _paths:
        if os.path.exists(os.path.join(_p, "da_common.py")):
            sys.path.insert(0, os.path.abspath(_p))
            break
    else:
        raise FileNotFoundError("da_common.py not found. See SETUP_kaggle.md for setup options.")
    from da_common import *

banner("Notebook 05 - Objectives 5 and 6")
spark = get_spark("05 product and customer")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### Objective 5 - Product-level analytics

In [ ]:
products = (df.groupBy("parent_asin", "Category")
            .agg(F.count("*").alias("reviews"),
                 F.round(F.avg("rating"), 3).alias("avg_rating"),
                 F.sum("helpful_vote").alias("total_helpful"),
                 F.round(F.avg("review_length"), 1).alias("avg_length"))).cache()

eligible = products.filter(F.col("reviews") >= MIN_REVIEWS).cache()
n_products, n_eligible = products.count(), eligible.count()
print(f"Products: {n_products:,}")
print(f"With at least {MIN_REVIEWS} reviews: {n_eligible:,} ({100*n_eligible/n_products:.2f}%)")

best_products = eligible.orderBy(F.desc("avg_rating"), F.desc("reviews")).limit(10)
worst_products = eligible.orderBy(F.asc("avg_rating"), F.desc("reviews")).limit(10)
most_reviewed = products.orderBy(F.desc("reviews")).limit(10)

print(f"\nHighest rated (n >= {MIN_REVIEWS})"); best_products.show(truncate=False)
print(f"Lowest rated (n >= {MIN_REVIEWS})");    worst_products.show(truncate=False)
print("Most reviewed");                          most_reviewed.show(truncate=False)

save_table(best_products, "obj5_best_products")
save_table(worst_products, "obj5_worst_products")
save_table(most_reviewed, "obj5_most_reviewed_products")
save_table(pd.DataFrame([{"products": n_products, "eligible_products": n_eligible,
                          "min_reviews": MIN_REVIEWS}]), "obj5_product_summary");

In [ ]:
mr, bp, wp = most_reviewed.toPandas(), best_products.toPandas(), worst_products.toPandas()
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

ax[0].barh(mr["parent_asin"][::-1], mr["reviews"][::-1], color=[ccol(c) for c in mr["Category"][::-1]])
ax[0].set_title("(a) Most-reviewed products"); ax[0].set_xlabel("Review count")

ax[1].barh(bp["parent_asin"][::-1], bp["avg_rating"][::-1], color="#16a34a")
ax[1].set_title(f"(b) Highest rated (n >= {MIN_REVIEWS})"); ax[1].set_xlabel("Average rating")
ax[1].set_xlim(max(0, bp["avg_rating"].min() - 0.25), 5.05)
for i, (v, n) in enumerate(zip(bp["avg_rating"][::-1], bp["reviews"][::-1])):
    ax[1].text(v, i, f"  {v} (n={n})", va="center", fontsize=8)

ax[2].barh(wp["parent_asin"][::-1], wp["avg_rating"][::-1], color="#ef4444")
ax[2].set_title(f"(c) Lowest rated (n >= {MIN_REVIEWS})"); ax[2].set_xlabel("Average rating")
ax[2].set_xlim(0, max(wp["avg_rating"].max() + 0.5, 1))
for i, (v, n) in enumerate(zip(wp["avg_rating"][::-1], wp["reviews"][::-1])):
    ax[2].text(v, i, f"  {v} (n={n})", va="center", fontsize=8)

plt.suptitle("Objective 5 - product-level rankings", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig07_product_rankings", "Objective 5 - product rankings with review counts")
plt.show()

### Objective 6 - Customer-activity analysis

In [ ]:
reviewer = (df.groupBy("user_id")
            .agg(F.count("*").alias("reviews"),
                 F.round(F.avg("rating"), 2).alias("avg_rating"),
                 F.sum("helpful_vote").alias("total_helpful"))).cache()
n_reviewers = reviewer.count()
print(f"Unique reviewers: {n_reviewers:,}")
print(f"Reviews per reviewer: {n_clean/n_reviewers:.2f}")

top_reviewers = reviewer.orderBy(F.desc("reviews"), F.desc("total_helpful")).limit(10)
print("\nMost active reviewers"); top_reviewers.show(truncate=False)

activity = (reviewer.withColumn("bucket",
                F.when(F.col("reviews") == 1, "1")
                 .when(F.col("reviews") == 2, "2")
                 .when(F.col("reviews").between(3, 5), "3-5")
                 .when(F.col("reviews").between(6, 10), "6-10")
                 .otherwise("11+"))
            .groupBy("bucket")
            .agg(F.count("*").alias("reviewers"), F.sum("reviews").alias("reviews_contributed")))

act = activity.toPandas()
act["bucket"] = pd.Categorical(act["bucket"], ["1", "2", "3-5", "6-10", "11+"], ordered=True)
act = act.sort_values("bucket").reset_index(drop=True)
act["pct_reviewers"] = (100 * act["reviewers"] / n_reviewers).round(2)
act["pct_reviews"] = (100 * act["reviews_contributed"] / n_clean).round(2)
print(act.to_string(index=False))

save_table(top_reviewers, "obj6_top_reviewers")
save_table(act, "obj6_activity_distribution")
save_table(pd.DataFrame([{"reviewers": n_reviewers,
                          "reviews_per_reviewer": round(n_clean / n_reviewers, 3)}]),
           "obj6_reviewer_summary");

In [ ]:
tr = top_reviewers.toPandas()
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
x = np.arange(len(act))
ax[0].bar(x, act["pct_reviewers"], color=ACCENT, label="% of reviewers")
ax[0].plot(x, act["pct_reviews"], color="#dc2626", marker="o", lw=2, label="% of all reviews")
ax[0].set_xticks(x); ax[0].set_xticklabels(act["bucket"].astype(str))
ax[0].set_title("Reviewer activity is highly skewed")
ax[0].set_xlabel("Reviews written by a user"); ax[0].set_ylabel("Percent"); ax[0].legend()
for i, v in enumerate(act["pct_reviewers"]):
    ax[0].text(i, v, f"{v}%", ha="center", va="bottom", fontsize=9)

ax[1].barh([u[:14] + "..." for u in tr["user_id"]][::-1], tr["reviews"][::-1], color="#7c3aed")
ax[1].set_title("Top 10 most active reviewers"); ax[1].set_xlabel("Reviews written")

plt.tight_layout()
savefig(fig, "fig08_reviewer_activity", "Objective 6 - reviewer activity distribution")
plt.show()

### Findings

Only a small fraction of products clear the fifty-review threshold, which is the whole reason the
threshold exists: without it the rating leaderboard would be filled by products with a handful of
reviews and no statistical meaning.

Reviewer activity follows the long-tail shape typical of user-generated content. The large
majority of reviewers contribute a single review, so the corpus is broad rather than deep, and
per-reviewer averages are noisy for almost everyone in it.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")